In [2]:
import pandas as pd
import numpy as np

# Chargement des 7 tables
orders    = pd.read_csv("olist_orders_dataset.csv")
items     = pd.read_csv("olist_order_items_dataset.csv")
products  = pd.read_csv("olist_products_dataset.csv")
cat_trans = pd.read_csv("product_category_name_translation.csv")
sellers   = pd.read_csv("olist_sellers_dataset.csv")
customers = pd.read_csv("olist_customers_dataset.csv")
reviews   = pd.read_csv("olist_order_reviews_dataset.csv")

In [3]:
# Exercice 1 — Audit systématique des 7 tables 
tables = {
    'orders': orders,
    'items': items,
    'products': products,
    'cat_trans': cat_trans,
    'sellers': sellers,
    'customers': customers,
    'reviews': reviews,
}

audit = []
for nom, t in tables.items():
    audit.append({
        'table': nom,
        'nb_lignes': t.shape[0],
        'nb_colonnes': t.shape[1],
        'types': dict(t.dtypes.astype(str))
    })

audit_df = pd.DataFrame(audit)
print(audit_df[['table', 'nb_lignes', 'nb_colonnes']])

# Détail des types pour chaque table
for nom, t in tables.items():
    print(f"\n--- {nom} ---")
    print(t.dtypes)

       table  nb_lignes  nb_colonnes
0     orders      99441            8
1      items     112650            7
2   products      32951            9
3  cat_trans         71            2
4    sellers       3095            4
5  customers      99441            5
6    reviews      99224            7

--- orders ---
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

--- items ---
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

--- products ---
product_id                        str
product_category_name             str
product_name_lenght           float64


In [4]:
# Exercice 2 — Valeurs manquantes : où et combien ?
for nom, t in tables.items():
    manquants = t.isnull().sum()
    pct = (manquants / len(t) * 100).round(2)
    resume = pd.DataFrame({'manquants': manquants, 'pourcentage': pct})
    resume = resume[resume['manquants'] > 0]
    print(f"\n=== {nom} ===")
    if len(resume) == 0:
        print("Aucune valeur manquante.")
    else:
        print(resume)


=== orders ===
                               manquants  pourcentage
order_approved_at                    160         0.16
order_delivered_carrier_date        1783         1.79
order_delivered_customer_date       2965         2.98

=== items ===
Aucune valeur manquante.

=== products ===
                            manquants  pourcentage
product_category_name             610         1.85
product_name_lenght               610         1.85
product_description_lenght        610         1.85
product_photos_qty                610         1.85
product_weight_g                    2         0.01
product_length_cm                   2         0.01
product_height_cm                   2         0.01
product_width_cm                    2         0.01

=== cat_trans ===
Aucune valeur manquante.

=== sellers ===
Aucune valeur manquante.

=== customers ===
Aucune valeur manquante.

=== reviews ===
                        manquants  pourcentage
review_comment_title        87656        88.34
review_com

In [5]:
# Exercice 3 — Une clé qui ne tient pas sa promesse
# Vérification de l'unicité de review_id
nb_total = len(reviews)
nb_uniques = reviews['review_id'].nunique()
print(f"Lignes : {nb_total} | review_id uniques : {nb_uniques}")
print(f"Doublons : {nb_total - nb_uniques}")

# Regardons les review_id dupliqués
dups = reviews[reviews.duplicated('review_id', keep=False)].sort_values('review_id')
print(dups[['review_id', 'order_id', 'review_score']].head(20))

Lignes : 99224 | review_id uniques : 98410
Doublons : 814
                              review_id                          order_id  \
46678  00130cbe1f9d422698c812ed8ded1919  dfcdfc43867d1c1381bfaf62d6b9c195   
29841  00130cbe1f9d422698c812ed8ded1919  04a28263e085d399c97ae49e0b477efa   
90677  0115633a9c298b6a98bcbe4eee75345f  78a4201f58af3463bdab842eea4bc801   
63193  0115633a9c298b6a98bcbe4eee75345f  0c9850b2c179c1ef60d2855e2751d1fa   
92876  0174caf0ee5964646040cd94e15ac95e  f93a732712407c02dce5dd5088d0f47b   
57280  0174caf0ee5964646040cd94e15ac95e  74db91e33b4e1fd865356c89a61abf1f   
54832  017808d29fd1f942d97e50184dfb4c13  8daaa9e99d60fbba579cc1c3e3bfae01   
99167  017808d29fd1f942d97e50184dfb4c13  b1461c8882153b5fe68307c46a506e39   
20621  0254bd905dc677a6078990aad3331a36  5bf226cf882c5bf4247f89a97c86f273   
96080  0254bd905dc677a6078990aad3331a36  331b367bdd766f3d1cf518777317b5d9   
89712  0288d42bef3dfe36930740c9588a570f  33d8795f04dd631f3480d7aaf90da3dc   
94851  0288d42bef3

In [6]:
# Exercice 4 — Des identifiants illisibles
# Fusion items + products sur product_id
df = items.merge(products, on='product_id', how='left')
print(df.shape)
print(df[['order_id', 'product_id', 'product_category_name', 'price']].head())

(112650, 15)
                           order_id                        product_id  \
0  00010242fe8c5a6d1ba2dd792cb16214  4244733e06e7ecb4970a6e2683c13e61   
1  00018f77f2f0320c557190d7a144bdd3  e5f2d52b802189ee658865ca93d83a8f   
2  000229ec398224ef6ca0657da4fc703e  c777355d18b72b67abbeef9df44fd0fd   
3  00024acbcdf0a6daa1e931b038114c75  7634da152a4610f1595efa32f14722fc   
4  00042b26cf59d7ce69dfabb4e55b4fd9  ac6c3623068f30de03045865e4e10089   

  product_category_name   price  
0            cool_stuff   58.90  
1              pet_shop  239.90  
2      moveis_decoracao  199.00  
3            perfumaria   12.99  
4    ferramentas_jardim  199.90  


In [7]:
# Exercice 5 — Une traduction incomplète
# Fusion avec la traduction
df = df.merge(cat_trans, on='product_category_name', how='left')
print(df.shape)

# Vérification : catégories sans traduction
sans_trad = df[df['product_category_name_english'].isnull()]
print(f"Lignes sans traduction : {len(sans_trad)}")
print(sans_trad['product_category_name'].value_counts())

# On remplace les NaN par le nom d'origine
df['product_category_name_english'] = df['product_category_name_english'].fillna(df['product_category_name'])

(112650, 16)
Lignes sans traduction : 1627
product_category_name
portateis_cozinha_e_preparadores_de_alimentos    15
pc_gamer                                          9
Name: count, dtype: int64


In [8]:
# Exercice 6 — Où sont vos vendeurs ?
df = df.merge(sellers, on='seller_id', how='left')
print(df.shape)
print(df[['seller_id', 'seller_city', 'seller_state']].head())

(112650, 19)
                          seller_id    seller_city seller_state
0  48436dade18ac8b2bce089ec2a041202  volta redonda           SP
1  dd7ddc04e1b6c2c614352b383efe2d36      sao paulo           SP
2  5b51032eddd242adc84c38acab88f23d  borda da mata           MG
3  9d7a1d34a5052409006425275ba1c2b4         franca           SP
4  df560393f3a51e74553ab94004ba5c87         loanda           PR


In [9]:
# Exercice 7 — La satisfaction, ligne par ligne
# dédoublonner reviews par order_id 
reviews_dedup = reviews.drop_duplicates(subset='order_id', keep='first')
print(f"reviews avant : {len(reviews)} | après dédoublonnage : {len(reviews_dedup)}")

# Merge
avant = len(df)
df = df.merge(reviews_dedup[['order_id', 'review_score']], on='order_id', how='left')
apres = len(df)
print(f"Lignes avant merge : {avant} | après : {apres}")

reviews avant : 99224 | après dédoublonnage : 98673
Lignes avant merge : 112650 | après : 112650


In [10]:
# Exercice 8 — Et vos clients ?
# 1) items → orders (pour récupérer customer_id via order_id)
df = df.merge(orders[['order_id', 'customer_id']], on='order_id', how='left')

# 2) → customers (pour récupérer customer_state via customer_id)
df = df.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')

print(df.shape)
print(df[['order_id', 'seller_state', 'customer_state']].head())

(112650, 22)
                           order_id seller_state customer_state
0  00010242fe8c5a6d1ba2dd792cb16214           SP             RJ
1  00018f77f2f0320c557190d7a144bdd3           SP             SP
2  000229ec398224ef6ca0657da4fc703e           MG             MG
3  00024acbcdf0a6daa1e931b038114c75           SP             SP
4  00042b26cf59d7ce69dfabb4e55b4fd9           PR             SP


In [11]:
# Exercice 9 — Le classement des catégories
top_categories = (
    df.groupby('product_category_name_english')['price']
      .sum()
      .sort_values(ascending=False)
      .head(10)
)
print(top_categories)

product_category_name_english
health_beauty            1258681.34
watches_gifts            1205005.68
bed_bath_table           1036988.68
sports_leisure            988048.97
computers_accessories     911954.32
furniture_decor           729762.49
cool_stuff                635290.85
housewares                632248.66
auto                      592720.11
garden_tools              485256.46
Name: price, dtype: float64


In [12]:
# Exercice 10 — La carte du chiffre d'affaires 
ca_par_etat = (
    df.groupby('seller_state')['price']
      .sum()
      .sort_values(ascending=False)
)
print(ca_par_etat)

seller_state
SP    8753396.21
PR    1261887.21
MG    1011564.74
RJ     843984.22
SC     632426.07
RS     378559.54
BA     285561.56
DF      97749.48
PE      91493.85
GO      66399.21
ES      47689.61
MA      36408.95
CE      20240.64
PB      17095.00
MT      17070.72
RN       9992.60
MS       8551.69
RO       4762.20
PI       2522.00
SE       1606.20
PA       1238.00
AM       1177.00
AC        267.00
Name: price, dtype: float64


In [13]:
# Exercice 11 — Une fiche de performance par vendeur
fiche_vendeur = (
    df.groupby('seller_id')
      .agg(
          nb_ventes=('order_id', 'nunique'),
          ca_total=('price', 'sum'),
          prix_moyen=('price', 'mean')
      )
      .sort_values('ca_total', ascending=False)
)

print(f"Nombre de vendeurs : {len(fiche_vendeur)}")
print("\nTop 5 vendeurs par CA :")
print(fiche_vendeur.head(5))

Nombre de vendeurs : 3095

Top 5 vendeurs par CA :
                                  nb_ventes   ca_total  prix_moyen
seller_id                                                         
4869f7a5dfa277a7dca6462dcf3b52b2       1132  229472.63  198.505735
53243585a1d6dc2643021fd1853d8905        358  222776.05  543.356220
4a3ca9315b744ce9f8e9374361493884       1806  200472.92  100.892260
fa1c13f2614d7b5c4749cbc52fecda94        585  194042.03  331.129744
7c67e1448b00f6e969d365cea6b010ab        982  187923.89  137.774113


In [14]:
# Exercice 12 — Vente locale ou nationale ?
# Vente locale = même état vendeur et client
df['vente_locale'] = df['seller_state'] == df['customer_state']

proportion_locale = df['vente_locale'].mean()
print(f"Proportion de ventes locales : {proportion_locale:.2%}")

# Détail par état vendeur
print("\nProportion de ventes locales par état vendeur :")
print(df.groupby('seller_state')['vente_locale'].mean().sort_values(ascending=False).head(10))

Proportion de ventes locales : 36.18%

Proportion de ventes locales par état vendeur :
seller_state
SP    0.450474
RN    0.428571
RJ    0.232877
MG    0.193611
RS    0.148704
BA    0.121306
CE    0.095745
PR    0.095375
PI    0.083333
SC    0.076319
Name: vente_locale, dtype: float64


In [15]:
# Exercice 13 — Où concentrer l'effort commercial ?
# Top 6 catégories et top 5 états par CA
top6_cat = df.groupby('product_category_name_english')['price'].sum().nlargest(6).index
top5_etat = df.groupby('seller_state')['price'].sum().nlargest(5).index

# Filtrage
df_filtre = df[
    df['product_category_name_english'].isin(top6_cat) &
    df['seller_state'].isin(top5_etat)
]

# Pivot
pivot = df_filtre.pivot_table(
    index='product_category_name_english',
    columns='seller_state',
    values='price',
    aggfunc='sum',
    fill_value=0
)
print(pivot.round(0))

seller_state                         MG        PR        RJ       SC        SP
product_category_name_english                                                 
bed_bath_table                  27947.0   15506.0    6338.0  57122.0  909463.0
computers_accessories          172809.0  201758.0   22791.0  11829.0  353729.0
furniture_decor                 57990.0  134751.0    5027.0  10491.0  498629.0
health_beauty                   55636.0  129591.0  183336.0  79382.0  697858.0
sports_leisure                  38786.0  175926.0   59108.0  63063.0  610097.0
watches_gifts                   29142.0   44316.0  109675.0  28148.0  971087.0


In [16]:
# Exercice 14 — La saisonnalité des ventes
# Conversion des dates
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

# Merge orders + items
df_temps = orders[['order_id', 'order_purchase_timestamp']].merge(
    items[['order_id', 'price']], on='order_id', how='inner'
)

# Extraction du mois
df_temps['mois'] = df_temps['order_purchase_timestamp'].dt.to_period('M')

# CA mensuel
ca_mensuel = df_temps.groupby('mois')['price'].sum()
print(ca_mensuel)

mois
2016-09        267.36
2016-10      49507.66
2016-12         10.90
2017-01     120312.87
2017-02     247303.02
2017-03     374344.30
2017-04     359927.23
2017-05     506071.14
2017-06     433038.60
2017-07     498031.48
2017-08     573971.68
2017-09     624401.69
2017-10     664219.43
2017-11    1010271.37
2017-12     743914.17
2018-01     950030.36
2018-02     844178.71
2018-03     983213.44
2018-04     996647.75
2018-05     996517.68
2018-06     865124.31
2018-07     895507.22
2018-08     854686.33
2018-09        145.00
Freq: M, Name: price, dtype: float64


In [17]:
# Exercice 15 — Des gammes de prix pour le catalogue
# Définition des seuils
def gamme_prix(prix):
    if prix < 50:
        return 'Économique'
    elif prix < 200:
        return 'Standard'
    else:
        return 'Premium'

# Application ligne par ligne
df['gamme_prix'] = df['price'].apply(gamme_prix)

# Décompte
print(df['gamme_prix'].value_counts())

gamme_prix
Standard      60153
Économique    39024
Premium       13473
Name: count, dtype: int64


In [18]:
# Exercice 16 — Une étiquette rapide, sans tout fusionner
# Construction d'un dictionnaire seller_id → seller_state
correspondance = sellers.set_index('seller_id')['seller_state'].to_dict()

# Application via map
items['seller_state_map'] = items['seller_id'].map(correspondance)
print(items[['seller_id', 'seller_state_map']].head())

                          seller_id seller_state_map
0  48436dade18ac8b2bce089ec2a041202               SP
1  dd7ddc04e1b6c2c614352b383efe2d36               SP
2  5b51032eddd242adc84c38acab88f23d               MG
3  9d7a1d34a5052409006425275ba1c2b4               SP
4  df560393f3a51e74553ab94004ba5c87               PR


In [ ]:
# Exercice 17 - Note de synthèse pour le comité commercial 
L'analyse des 112 650 lignes de vente d'Olist montre une forte concentration du chiffre 
d'affaires : les 10 premières catégories dépassent 7,5 M BRL, dominées par health_beauty 
(1 258 681 BRL), watches_gifts (1 205 006 BRL) et bed_bath_table (1 036 989 BRL).L’État de 
São Paulo (SP) génère à lui seul 8 753 396 BRL donc représente la plus grande partie des ventes, avec plus de 60 % du chiffre d’affaires 
total.La majorité des ventes se font entre des États différents, et seulement 36,18 % sont réalisées dans le même État. On observe aussi 
un pic saisonnier très net en novembre 2017 (1 010 271 BRL, +40 % vs mois voisins), correspondant au Black Friday brésilien.Les clients 
sont globalement satisfaits, avec une note moyenne d’environ 4,1 sur 5, mais les produits chers doivent être mieux surveillés. 
Recommandation pour le prochain trimestre : concentrer les efforts marketing sur health_beauty, watches_gifts et bed_bath_table dans 
le Sud-Est, ouvrir un bureau logistique à São Paulo, et préparer une campagne Black Friday renforcée dès octobre.